# ⚡ EEQ401: Autonomous Contingency Remediation Agent (Kaggle GPU Runner)

This notebook runs the **DQN Agent** and **Deep Grid Security Classifier** on Kaggle's free **NVIDIA GPU** (T4 / P100).

### Features:
1. **Automatic GPU Acceleration:** Utilizes CUDA for tensor operations, reducing run time from ~4 hours to ~30-40 minutes.
2. **Zero Code Duplication:** Seamlessly imports the modular repository.
3. **Interactive Visualizations:** Generates loss curves, safe rate plots, and remediation analytics.

In [ ]:
# 1. Check GPU Availability
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Install Dependencies
!pip install -q pypower scipy matplotlib seaborn pandas tqdm scikit-learn

In [ ]:
# 3. Clone / Setup Repository (or run from uploaded Kaggle working directory)
import os

# If running as a standalone Kaggle notebook with git repo:
if not os.path.exists('src'):
    !git clone https://github.com/Ajayramaswamy2005/contingency_agent-2.git repo
    %cd repo
else:
    print("Repository files already present in working directory.")

In [ ]:
# 4. Train Deep Neural Classifier on 100k Dataset (GPU Accelerated)
!python train_classifier_100k.py

In [ ]:
# 5. Train DQN Agent on 100k Dataset with GPU Acceleration (80,000 Episodes)
!python train_on_new_samples.py --episodes 80000 --device cuda --checkpoint-freq 10000

In [ ]:
# 6. Run Complete End-to-End Pipeline Demonstration
!python run_full_pipeline.py --device cuda --num-scenarios 50 --verbose --output-csv kaggle_pipeline_results.csv

In [ ]:
# 7. Cross-Dataset Validation (200k & 300k Unseen Datasets)
!python validate_cross_dataset.py

In [ ]:
# 8. Visualize Training Convergence and Security Remediation Results
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Load training history
history_path = "models/new_samples_dqn/training_history.csv"
if os.path.exists(history_path):
    df_hist = pd.read_csv(history_path)
    axes[0].plot(df_hist['episode'], df_hist['loss'], color='crimson', lw=1.5)
    axes[0].set_title("DQN Bellman Loss vs Episodes", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Huber Loss")
    
    axes[1].plot(df_hist['episode'], df_hist['safe_rate_100'], color='teal', lw=1.5)
    axes[1].set_title("Rolling Safe Rate (%) vs Episodes", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Safe Rate (%)")
    axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig("training_performance_curves.png", dpi=300)
plt.show()

# Print Evaluation Summary
summary_path = "models/new_samples_dqn/evaluation_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("\n" + "="*50)
    print("  FINAL BENCHMARK EVALUATION SUMMARY")
    print("="*50)
    for k, v in summary.items():
        print(f"  {k:<35}: {v}")
    print("="*50)

In [ ]:
# 9. Download Trained Artifacts
!tar -czvf trained_models_and_metrics.tar.gz models/ *.png *.csv
print("Artifacts archived successfully to trained_models_and_metrics.tar.gz!")